## 1. Load and Preprocess WBC (original)

In [1]:
import sys
sys.path.append("../")
from utils.fetch_bcw_dataset import fetch_local_bcw
import pandas as pd

In [2]:
bcw_df = fetch_local_bcw("../data/Breast Cancer Wisconsin.csv")

In [3]:
print(bcw_df.isna().any())
print(f"\nTotal na line:\n{bcw_df.isna().sum()}")

Clump_thickness                False
Uniformity_of_cell_size        False
Uniformity_of_cell_shape       False
Marginal_adhesion              False
Single_epithelial_cell_size    False
Bare_nuclei                     True
Bland_chromatin                False
Normal_nucleoli                False
Mitoses                        False
Class                          False
dtype: bool

Total na line:
Clump_thickness                 0
Uniformity_of_cell_size         0
Uniformity_of_cell_shape        0
Marginal_adhesion               0
Single_epithelial_cell_size     0
Bare_nuclei                    16
Bland_chromatin                 0
Normal_nucleoli                 0
Mitoses                         0
Class                           0
dtype: int64


In [4]:
nona_bcw_df = bcw_df[~bcw_df['Bare_nuclei'].isna()]

In [5]:
nona_bcw_df.dtypes

Clump_thickness                  int64
Uniformity_of_cell_size          int64
Uniformity_of_cell_shape         int64
Marginal_adhesion                int64
Single_epithelial_cell_size      int64
Bare_nuclei                    float64
Bland_chromatin                  int64
Normal_nucleoli                  int64
Mitoses                          int64
Class                            int64
dtype: object

In [6]:
from src.preprocessing import foolish_discretization

In [7]:
discretized_bare_nuclei = foolish_discretization(nona_bcw_df['Bare_nuclei'])
assert discretized_bare_nuclei.tolist() == discretized_bare_nuclei.copy().astype("int64").tolist()

In [8]:
clear_bcw_df = fetch_local_bcw("../data/Clear Breast Cancer Wisconsin.csv")
clear_bcw_df.dtypes

Clump_thickness                int64
Uniformity_of_cell_size        int64
Uniformity_of_cell_shape       int64
Marginal_adhesion              int64
Single_epithelial_cell_size    int64
Bare_nuclei                    int64
Bland_chromatin                int64
Normal_nucleoli                int64
Mitoses                        int64
Class                          int64
dtype: object

## 2. Reduction and Calculation of Core

In [9]:
import numpy as np

In [10]:
U = clear_bcw_df
C = list(U.columns[0:-1])
V = [c for c in U.columns if c not in C]
D = U[V].drop_duplicates().to_numpy()

The following codes aim to debug.
While you replace the statement in body of under_appr: 
`return X.loc[list(under_appr_ind_set)]` into 
`return X.iloc[list(under_appr_ind_set)]`,
you will encounter IndexError.

```python
under_appr_ind_set = set()
for _, D_i in U_div_D.items():
    D_1 = D_i
    break
X_ind_set = set(D_1.index)
for cls_set in equiv_cls.values():
    if not cls_set.issubset(X_ind_set): continue
    under_appr_ind_set = under_appr_ind_set.union(cls_set)
for i in list(under_appr_ind_set):
    try:
        D_1.loc[[i]]
        # print(f"{i} works!")
    except IndexError as ie:
        print(f"IndexError: {i}")
```

In [11]:
def equiv_cls(B):
    equiv_cls_B = dict()
    for r in U.groupby(B, group_keys=True).apply(lambda x: x).index:
        if r[0:-1] not in equiv_cls_B.keys():
            equiv_cls_B[r[0:-1]] = [r[-1]]
            continue
        equiv_cls_B[r[0:-1]].append(r[-1])
    return {k: set(equiv_cls_B[k]) for k in equiv_cls_B.keys()}

def under_appr(B, X) -> pd.DataFrame:
    under_appr_ind_set = set()
    X_ind_set = set(X.index)
    for cls_set in equiv_cls(B).values():
        if not cls_set.issubset(X_ind_set): continue
        under_appr_ind_set = under_appr_ind_set.union(cls_set)
    return X.loc[list(under_appr_ind_set)]

def pos(B) -> pd.DataFrame:
    _U_div_D = {tuple(d): U[(U[V]==d).all(axis=1)] for d in D}
    under_appr_dict = {k: under_appr(B, D_i) for k, D_i in _U_div_D.items()}
    return pd.concat(under_appr_dict.values())

def gamma(B): return len(pos(B)) / len(U)

In [12]:
from random import randint
B = []
for _ in range(7):
    B.append(C[randint(0, len(C)-1)])
B = list(set(B))
B

['Marginal_adhesion',
 'Bland_chromatin',
 'Normal_nucleoli',
 'Mitoses',
 'Uniformity_of_cell_size']

In [13]:
gamma(B)

0.986822840409956

In [14]:
def print_B_g(B):
    for c in set(U.columns[0:-1]).difference(B):
        print(f"{B.union({c})}: {gamma(list(B.union({c})))}")

In [15]:
B = set()
print_B_g(B)

{'Marginal_adhesion'}: 0.06149341142020498
{'Bland_chromatin'}: 0.08638360175695461
{'Bare_nuclei'}: 0.0
{'Normal_nucleoli'}: 0.10980966325036604
{'Mitoses'}: 0.0424597364568082
{'Uniformity_of_cell_shape'}: 0.0951683748169839
{'Clump_thickness'}: 0.12152269399707175
{'Uniformity_of_cell_size'}: 0.17862371888726208
{'Single_epithelial_cell_size'}: 0.0029282576866764276


In [16]:
B = {'Uniformity_of_cell_size'}
print_B_g(B)

{'Marginal_adhesion', 'Uniformity_of_cell_size'}: 0.3323572474377745
{'Bland_chromatin', 'Uniformity_of_cell_size'}: 0.636896046852123
{'Bare_nuclei', 'Uniformity_of_cell_size'}: 0.7862371888726208
{'Normal_nucleoli', 'Uniformity_of_cell_size'}: 0.33528550512445093
{'Mitoses', 'Uniformity_of_cell_size'}: 0.22693997071742314
{'Uniformity_of_cell_shape', 'Uniformity_of_cell_size'}: 0.29282576866764276
{'Clump_thickness', 'Uniformity_of_cell_size'}: 0.7818448023426061
{'Uniformity_of_cell_size', 'Single_epithelial_cell_size'}: 0.2913616398243045


In [17]:
B = {'Uniformity_of_cell_size', 'Bare_nuclei'}
print_B_g(B)

{'Marginal_adhesion', 'Bare_nuclei', 'Uniformity_of_cell_size'}: 0.9092240117130308
{'Bland_chromatin', 'Bare_nuclei', 'Uniformity_of_cell_size'}: 0.9458272327964861
{'Normal_nucleoli', 'Bare_nuclei', 'Uniformity_of_cell_size'}: 0.9677891654465594
{'Mitoses', 'Bare_nuclei', 'Uniformity_of_cell_size'}: 0.8887262079062958
{'Uniformity_of_cell_shape', 'Bare_nuclei', 'Uniformity_of_cell_size'}: 0.9458272327964861
{'Bare_nuclei', 'Clump_thickness', 'Uniformity_of_cell_size'}: 0.9853587115666179
{'Bare_nuclei', 'Uniformity_of_cell_size', 'Single_epithelial_cell_size'}: 0.9619326500732065


In [18]:
B = {'Clump_thickness', 'Uniformity_of_cell_size', 'Bare_nuclei'}
print_B_g(B)

{'Marginal_adhesion', 'Bare_nuclei', 'Clump_thickness', 'Uniformity_of_cell_size'}: 0.9970717423133236
{'Bland_chromatin', 'Bare_nuclei', 'Clump_thickness', 'Uniformity_of_cell_size'}: 1.0
{'Normal_nucleoli', 'Bare_nuclei', 'Clump_thickness', 'Uniformity_of_cell_size'}: 1.0
{'Mitoses', 'Bare_nuclei', 'Clump_thickness', 'Uniformity_of_cell_size'}: 0.9882869692532943
{'Uniformity_of_cell_shape', 'Bare_nuclei', 'Clump_thickness', 'Uniformity_of_cell_size'}: 0.9970717423133236
{'Bare_nuclei', 'Clump_thickness', 'Uniformity_of_cell_size', 'Single_epithelial_cell_size'}: 0.9970717423133236


In [19]:
import pickle

In [20]:
def reduce(C, verbose=True, save_path=None):
    def _save(cache):
        with open(save_path, 'wb') as f:
            pickle.dump(cache, f)
    def _find_argmax(C, B):
        gen = ((gamma(list(B.union({c}))), c) for c in set(C).difference(B))
        g, argmax_c = 0, ...
        if save_path: _cache = {}
        for _ in range(len(set(C).difference(B))):
            g_, c_ = next(gen)
            if verbose: print(f"{B.union({c_})}: {g_}")
            if save_path: _cache[tuple(B.union({c_}))] = g_
            if g_ == 1: 
                g, argmax_c = g_, c_
                break
            if g_ <= g: continue
            g, argmax_c = g_, c_
        if save_path: return g, argmax_c, _cache
        return g, argmax_c
    B_ = set()
    if save_path: cache = {}
    while len(B_) < len(C):
        t_ = _find_argmax(C, B_)
        B_.update({t_[1]})
        if verbose: print(f"{B_}\n")
        if save_path: cache.update(t_[2])
        if t_[0] == 1: break
    if save_path: _save(cache)
    return B_

In [21]:
import time
from datetime import datetime

In [22]:
file_path = f"../data/reduce/reduce_2026-10-22.pkl"
start = time.time()
print(reduce(C))
end = time.time()
print(f"Total: {end-start}")

{'Marginal_adhesion'}: 0.06149341142020498
{'Bland_chromatin'}: 0.08638360175695461
{'Bare_nuclei'}: 0.0
{'Normal_nucleoli'}: 0.10980966325036604
{'Mitoses'}: 0.0424597364568082
{'Uniformity_of_cell_shape'}: 0.0951683748169839
{'Clump_thickness'}: 0.12152269399707175
{'Uniformity_of_cell_size'}: 0.17862371888726208
{'Single_epithelial_cell_size'}: 0.0029282576866764276
{'Uniformity_of_cell_size'}

{'Marginal_adhesion', 'Uniformity_of_cell_size'}: 0.3323572474377745
{'Bland_chromatin', 'Uniformity_of_cell_size'}: 0.636896046852123
{'Bare_nuclei', 'Uniformity_of_cell_size'}: 0.7862371888726208
{'Normal_nucleoli', 'Uniformity_of_cell_size'}: 0.33528550512445093
{'Mitoses', 'Uniformity_of_cell_size'}: 0.22693997071742314
{'Uniformity_of_cell_shape', 'Uniformity_of_cell_size'}: 0.29282576866764276
{'Clump_thickness', 'Uniformity_of_cell_size'}: 0.7818448023426061
{'Uniformity_of_cell_size', 'Single_epithelial_cell_size'}: 0.2913616398243045
{'Bare_nuclei', 'Uniformity_of_cell_size'}

{'Marg

In [23]:
gamma(['Bland_chromatin', 'Clump_thickness', 'Bare_nuclei', 'Uniformity_of_cell_size'])

1.0

In [24]:
with open(file_path, 'rb') as f:
    state = pickle.load(f)
state

{('Bland_chromatin',): 0.08638360175695461,
 ('Uniformity_of_cell_size',): 0.17862371888726208,
 ('Mitoses',): 0.0424597364568082,
 ('Bare_nuclei',): 0.0,
 ('Clump_thickness',): 0.12152269399707175,
 ('Normal_nucleoli',): 0.10980966325036604,
 ('Single_epithelial_cell_size',): 0.0029282576866764276,
 ('Uniformity_of_cell_shape',): 0.0951683748169839,
 ('Marginal_adhesion',): 0.06149341142020498,
 ('Uniformity_of_cell_size', 'Bland_chromatin'): 0.636896046852123,
 ('Mitoses', 'Uniformity_of_cell_size'): 0.22693997071742314,
 ('Uniformity_of_cell_size', 'Bare_nuclei'): 0.7862371888726208,
 ('Uniformity_of_cell_size', 'Clump_thickness'): 0.7818448023426061,
 ('Uniformity_of_cell_size', 'Normal_nucleoli'): 0.33528550512445093,
 ('Uniformity_of_cell_size',
  'Single_epithelial_cell_size'): 0.2913616398243045,
 ('Uniformity_of_cell_shape', 'Uniformity_of_cell_size'): 0.29282576866764276,
 ('Uniformity_of_cell_size', 'Marginal_adhesion'): 0.3323572474377745,
 ('Uniformity_of_cell_size',
  'Ba

In [25]:
# TODO: To implement
# calculate the core in power set of C
# by using gamma_A \leq gamma \gamma_B
# if A \subseteq B
def core_v1(C):
    def reducible_or_not(c, C):
        raise NotImplementedError()
        for _c in C.difference({c}):
            ...        
    _core = set()
    for c in C:
        if reducible_or_not(c, C):
           continue
        s.add(c)
    return _core

# TODO: To implement
# calculate the core during reduction
def core_v2(C, verbose=True):
    raise NotImplementedError()
    def _find_argmax(C, B):
        gen = ((gamma(list(B.union({c}))), c) for c in set(C).difference(B))
        g, argmax_c = 0, ...
        for _ in range(len(set(C).difference(B))):
            g_, c_ = next(gen)
            if verbose: print(f"{B.union({c_})}: {g_}")
            if g_ == 1: 
                g, argmax_c = g_, c_
                break
            if g_ <= g: continue
            g, argmax_c = g_, c_
        return g, argmax_c
    B_ = set()
    while len(B_) < len(C):
        t_ = _find_argmax(C, B_)
        B_.update({t_[1]})
        if verbose: print(f"{B_}\n")
        if t_[0] == 1: break
    return B_

In [26]:
def time_core(core_method):
    start = time.time()
    print(core_method(C))
    end = time.time()
    print(f"Total: {end-start}")

In [27]:
time_core(core_v1)
time_core(core_v2)

NotImplementedError: 